# Business Analytics 2, Week 7: Hypothesis Testing

Companion notebook for the Week 7 page of the BA2 course app: p-values, the one-sample z-test, and comparing two groups (two averages and two rates).

**Run the cells in order.** Cells 2–4 reuse `math` and `norm` from cell 2, and cell 6 reuses `df` from cell 5. If you restart the runtime, run them again from the top.

Cell 5 reads `jam-tasting.csv` from the course's GitHub repository. If that link ever fails, download the file from the Week 7 page, upload it with the Files panel on the left, and change `BASE` to `""`.

_Generated from the code blocks on the Week 7 page._

## 1. The coin: how surprising is 9 heads in 10?

In [ ]:
from math import comb

# A fair coin, flipped 10 times: how often do we see 9 or 10 heads?
p = sum(comb(10, k) for k in (9, 10)) / 2**10
print(round(p, 4))   # 0.0107: about 1 time in 100

## 2. The lesson's video example (one-sample z-test)

In [ ]:
import math
from scipy.stats import norm

# The lesson's video example: H0 mean <= 7.5, Ha mean > 7.5 (right-tailed)
z = (8.2 - 7.5) / (2.1 / math.sqrt(35))
p = 1 - norm.cdf(z)
print(round(z, 2), round(p, 4))   # 1.97 0.0243  (the z table gives 0.0244 because it rounds z to 1.97)

## 3. Your turn: the coaching institute

In [ ]:
z_critical = norm.ppf(0.95)              # right-tailed, alpha = 0.05
z = (90 - 82) / (20 / math.sqrt(81))
p = 1 - norm.cdf(z)
print(round(z_critical, 3), round(z, 2), round(p, 5))   # 1.645 3.6 0.00016

## 4. Worked example: are the jars under-filled?

In [ ]:
# Supplier: 340 g per jar, sigma = 6 g. We weighed 36 jars: average 338 g.
# H0: mean >= 340   Ha: mean < 340   (left-tailed)
z = (338 - 340) / (6 / math.sqrt(36))
p = norm.cdf(z)                          # left tail
print(z, round(p, 4))                    # -2.0 0.0228 -> below 0.05, reject H0

## 5. Two groups: do tasting days sell more jars?

In [ ]:
import pandas as pd
from scipy import stats

BASE = "https://raw.githubusercontent.com/PA-WiT/BA2/main/public/data/"   # or upload the CSV and use its file name
df = pd.read_csv(BASE + "jam-tasting.csv")
yes = df.loc[df["tasting"] == "yes", "jars"]
no = df.loc[df["tasting"] == "no", "jars"]
print(yes.mean(), no.mean())             # 64.9 55.4

# Ha: tasting days sell MORE, so the test is right-tailed (alternative="greater")
t, p = stats.ttest_ind(yes, no, equal_var=False, alternative="greater")   # Welch's t-test
print(round(t, 2), round(p, 4))          # 2.55 0.0103

## 6. Two groups: does tasting turn more visitors into buyers?

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

g = df.groupby("tasting")[["buyers", "visitors"]].sum()
print((g["buyers"] / g["visitors"]).round(3))       # no 0.271, yes 0.308

z, p = proportions_ztest(g.loc[["yes", "no"], "buyers"], g.loc[["yes", "no"], "visitors"], alternative="larger")
print(round(z, 2), round(p, 4))          # 2.21 0.0134  (right-tailed: tasting converts MORE visitors)